# Lab 16: Speech to Text

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
Until now the Jetson used its **eyes**. In this lab it uses its **ears**. You record your voice with the microphone of the webcam, look at the sound as numbers, and let an AI model called **Whisper** turn it into **text**. This is **speech to text** (STT), also called speech recognition. You measure how accurate and how fast it is, and you build a **voice note** program that writes what you say into a log and reacts to command words.

**By the end of this lab you can:**
1. Record sound on the Jetson and explain sample rate, samples and level.
2. Look at a sound as a waveform and tell silence from speech.
3. Turn speech into text with Whisper and read the speed as a real-time factor.
4. Measure accuracy with the word error rate (WER).
5. Build a voice-note program that logs text and reacts to command words.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Press **Ctrl + S** often. Recording cells ask you to speak: read the text on the screen when it says **SPEAK NOW**.

> **Privacy.** The microphone records everything it hears. Record only yourself and classmates who agree. Delete recordings you do not need. The sound is processed on the Jetson and does not leave it, except for the first download of the model.

## 2. Background

**Sound as numbers.** A microphone turns the air pressure into a voltage. The Jetson **samples** it: it measures the voltage many times a second and stores each measure as a number. The number of measures per second is the **sample rate**. Speech needs 16 000 samples per second (16 kHz). One second of sound is a list of 16 000 numbers between -1 and 1.

| Word | Meaning | In this lab |
|---|---|---|
| Sample | One measure of the sound | One number |
| Sample rate | Samples per second | 16 000 |
| Mono | One microphone channel | Yes |
| Level (dB) | How loud the sound is, in decibels. A quiet room is about -60 dB, speaking is about -30 to -15 dB, 0 dB is the maximum. | `level_db` |

**Speech to text with Whisper.** Whisper is an AI model made by OpenAI. It was trained on hundreds of thousands of hours of speech. It listens to up to 30 seconds of sound and writes the words. This lab runs it on the Jetson with the program **faster-whisper**, on the CPU, so nothing is sent to the internet.

| Model | Size | Speed | Accuracy |
|---|---|---|---|
| `tiny.en` | 75 MB | Fastest | Lower |
| `base.en` | 145 MB | Fast | Good (the default of this lab) |
| `small.en` | 480 MB | Slower | Better |

The `.en` models know English only. The models without `.en` (`tiny`, `base`, `small`) know about 100 languages, including Malay (`ms`).

**Real-time factor (RTF).** `RTF = processing time / length of the sound`. An RTF of 0.5 means 10 seconds of speech take 5 seconds to turn into text. Below 1 is fast enough for a voice assistant.

**Word error rate (WER).** To measure accuracy, compare what the model wrote (the *hypothesis*) with what was said (the *reference*). The WER counts the words that were **substituted**, **deleted** or **inserted**:

`WER = (substitutions + deletions + insertions) / number of words in the reference`

A WER of 0 % is perfect. 10 % means about one wrong word in ten. The WER is calculated with the **edit distance**, the smallest number of word changes that turn one sentence into the other.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(16)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Set Up the Microphone (30 min)

### Step 1: Install the Parts
The cell checks `arecord` (records sound), `aplay` (plays sound) and the Python program `faster-whisper`. Setup of a new Jetson installs `alsa-utils`. If `faster-whisper` is missing, the cell installs it (about 1 minute, it needs the internet).

In [ ]:
import csv
import glob
import os
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime

import numpy as np

import labview

print("This notebook is saved in:", os.getcwd())
print("arecord:", shutil.which("arecord") or "MISSING (run: sudo apt install alsa-utils)")
print("aplay:  ", shutil.which("aplay") or "MISSING (run: sudo apt install alsa-utils)")
try:
    import faster_whisper
    print("faster-whisper is installed.")
except ImportError:
    print("Installing faster-whisper ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper"], check=True)
    print("faster-whisper is installed.")

### Step 2: Find Your Microphone
A webcam has a microphone. The cell lists the sound devices of the Jetson. Look for your webcam under **Microphones**. The line `card 2: ... device 0` is written `plughw:2,0` (card number, device number). Your teacher can tell you which one is yours.

In [ ]:
labview.audio_devices()

Set your microphone (change the numbers to your card and device) and record 3 seconds. The program prints the level. The cell **saves your choice**, so later labs use it too. If your speaker is not the default speaker, set it with `labview.speaker_set("plughw:1,0")`.

In [ ]:
MIC = "plughw:2,0"          # change this to your microphone
labview.mic_set(MIC)

print("Say 'one two three' after the message ...")
time.sleep(1)
print(">>> SPEAK NOW <<<")
sound = labview.record(3)
print("Done.")


def level_db(x):
    """The loudness of a sound in decibels (0 is the maximum, a quiet room is about -60)."""
    return 20 * np.log10(max(float(np.sqrt(np.mean(np.square(x)))), 1e-6))


print(f"Samples: {len(sound)}   seconds: {len(sound) / 16000:.1f}   level: {level_db(sound):.0f} dB")

Play it back. A player appears under the cell and plays on your **laptop**. The sound also plays on the speaker of the **Jetson** if one is connected.

In [ ]:
labview.play(sound)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Microphone name you set:
- Level of your voice in dB:
- Is your recording clear? What can you hear (noise, echo, other voices)?:
- How many samples are there in 3 seconds at 16 000 samples per second?:


---
# Part B: Look at Sound (35 min)

### Step 1: Draw the Waveform
A waveform is a picture of the sound: the horizontal line is time and the height is the loudness. The function `draw_wave` makes the picture with OpenCV (the tool of Lab 6). Run it for your recording. You see the loud parts (speech) and the flat parts (silence).

In [ ]:
import cv2
from IPython.display import Image, display


def draw_wave(x, width=900, height=200, rate=16000):
    """A picture of the sound: for each column the lowest and the highest sample."""
    picture = np.full((height, width, 3), 255, np.uint8)
    middle = height // 2
    cv2.line(picture, (0, middle), (width, middle), (200, 200, 200), 1)
    step = max(1, len(x) // width)
    for column in range(width):
        part = x[column * step:(column + 1) * step]
        if len(part):
            cv2.line(picture, (column, int(middle - part.max() * middle)), (column, int(middle - part.min() * middle)), (160, 90, 0), 1)
    seconds = len(x) / rate
    for second in range(int(seconds) + 1):
        column = int(second / seconds * width) if seconds else 0
        cv2.putText(picture, f"{second}s", (min(column + 2, width - 25), height - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (90, 90, 90), 1)
    return picture


display(Image(data=cv2.imencode(".png", draw_wave(sound))[1].tobytes()))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Can you see where you said each word?:
- How tall is the waveform when you are silent and when you speak?:


### Step 2: Speech or Silence?
The simplest speech detector measures the loudness of short pieces (20 milliseconds) and compares it with a limit. The function `speech_parts` returns the start and the end (in seconds) of every part that is louder than `LIMIT_DB`. This is useful: a program should listen to speech and ignore silence. Lab 17 uses the same idea.

In [ ]:
LIMIT_DB = -38              # louder than this counts as speech. Change it for your room.


def speech_parts(x, rate=16000, limit_db=LIMIT_DB, frame=0.02, gap=0.3):
    """The parts of the sound that are louder than limit_db: a list of (start, end) in seconds.
    Parts that are less than `gap` seconds apart are joined (a short pause between words)."""
    size = int(rate * frame)
    loud = [level_db(x[i:i + size]) > limit_db for i in range(0, len(x) - size, size)]
    parts, start = [], None
    for index, flag in enumerate(loud + [False]):
        if flag and start is None:
            start = index
        elif not flag and start is not None:
            end = index
            if parts and (start - parts[-1][1]) * frame < gap:
                parts[-1] = (parts[-1][0], end)
            else:
                parts.append((start, end))
            start = None
    return [(round(a * frame, 2), round(b * frame, 2)) for a, b in parts]


print("Level of the quiet room:", f"{level_db(sound[:4000]):.0f} dB", "(first quarter second)")
print("Speech parts (start, end in seconds):", speech_parts(sound))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Speech parts found:
- Is the quiet room level below your `LIMIT_DB`? If not, what do you change?:
- What happens to the speech parts if you set LIMIT_DB = -20, and if you set it to -55?:


---
# Part C: Speech to Text with Whisper (45 min)

### Step 1: Load the Model
The first time, the model is **downloaded** (it needs the internet, 145 MB for `base.en`, about a minute). It is kept for next time. `compute_type="int8"` makes the model smaller and faster on the CPU.

In [ ]:
# ---------------- Settings you can change ----------------
MODEL_SIZE = "base.en"      # "tiny.en" (faster), "base.en", "small.en" (better); without .en for other languages
LANGUAGE = "en"             # "en" for English, "ms" for Malay (use a model without .en)
# ----------------------------------------------------------
from faster_whisper import WhisperModel

started = time.time()
stt = WhisperModel(MODEL_SIZE, device="cpu", compute_type="int8", cpu_threads=4)
print(f"Model {MODEL_SIZE} loaded in {time.time() - started:.0f} seconds")

### Step 2: Turn Your Speech into Text
The function `transcribe` gives the text, the time it took and the real-time factor. Read the sentence when it says **SPEAK NOW**. Say: *The Jetson can hear and understand me.*

In [ ]:
def transcribe(x, language=None):
    """Speech to text. Returns (text, seconds it took, real-time factor)."""
    started = time.time()
    segments, info = stt.transcribe(x.astype(np.float32), language=language or LANGUAGE, beam_size=1)
    text = " ".join(s.text.strip() for s in segments).strip()
    took = time.time() - started
    return text, took, took / (len(x) / 16000)


print("Get ready ...")
time.sleep(1)
print(">>> SPEAK NOW: The Jetson can hear and understand me. <<<")
sound = labview.record(5)
text, took, rtf = transcribe(sound)
print()
print("Heard:", repr(text))
print(f"Took {took:.1f} s for {len(sound) / 16000:.0f} s of sound, real-time factor {rtf:.2f}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What did Whisper write?:
- Was it correct? Which words were wrong?:
- Real-time factor:
- Is the real-time factor below 1? What does it mean?:


### Step 3: Words and Time
Whisper also knows **when** each part was said. The cell prints each segment with its start and end time. It reads the file you just recorded, no need to speak again.

In [ ]:
segments, info = stt.transcribe(sound.astype(np.float32), language=LANGUAGE, beam_size=1)
print("Language:", info.language, "| sound length:", round(info.duration, 1), "s")
for s in segments:
    print(f"  {s.start:5.1f} s to {s.end:5.1f} s   {s.text.strip()}")

### Step 4: Measure the Accuracy (WER)
The function `wer` calculates the word error rate with the edit distance. First check it with sentences where you know the answer.

In [ ]:
def words_of(text):
    """Lower case words without punctuation, so 'Hello,' and 'hello' are the same word."""
    return re.sub(r"[^a-z0-9' ]", " ", text.lower()).split()


def wer(reference, hypothesis):
    """The word error rate (0 to 1 or more): edit distance of the words divided by the number of reference words."""
    ref, hyp = words_of(reference), words_of(hypothesis)
    previous = list(range(len(hyp) + 1))
    for i, r in enumerate(ref, 1):
        current = [i]
        for j, h in enumerate(hyp, 1):
            current.append(min(previous[j] + 1,                 # a word was deleted
                               current[j - 1] + 1,              # a word was inserted
                               previous[j - 1] + (r != h)))     # a word was changed (or is right)
        previous = current
    return previous[-1] / max(1, len(ref))


print(wer("the cat sat on the mat", "the cat sat on the mat"), "(expected 0.0)")
print(wer("the cat sat on the mat", "the cat sat on a mat"), "(expected about 0.17: one word of six)")
print(wer("the cat sat on the mat", "the cat sat"), "(expected 0.5: three words missing)")
print(wer("Turn on the light.", "turn on the light"), "(expected 0.0: capitals and dots are ignored)")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did all four lines give the expected numbers?:
- Why does the function make the text lower case and remove dots before it compares?:


Now measure your own speech. Read each sentence when it says **SPEAK NOW** (5 seconds each). The cell calculates the WER of each sentence and the average.

In [ ]:
SENTENCES = [
    "turn on the green light",
    "what is the temperature in the room",
    "the quick brown fox jumps over the lazy dog",
    "please count the people at the door",
    "start the camera and stop the music",
]

scores = []
for sentence in SENTENCES:
    print()
    print(f">>> SPEAK NOW: {sentence} <<<")
    heard = transcribe(labview.record(5))[0]
    scores.append(wer(sentence, heard))
    print("Heard:", repr(heard), f"-> WER {100 * scores[-1]:.0f} %")
print()
print(f"Average WER: {100 * sum(scores) / len(scores):.0f} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- WER of each of the five sentences:
- Average WER:
- Which words went wrong? Are they hard, short, or similar to other words?:


---
# Part D: Build a Voice Note Program (40 min)

### Step 1: The Program
The program records a short piece of speech, writes the text, saves it in a log, and reacts to **command words**. It is the base of a voice assistant (Lab 22).

| Function | What it does |
|---|---|
| `listen_once(seconds)` | Records, checks that there is speech (`speech_parts`), turns it into text. Returns `""` for silence, so Whisper does not invent words. |
| `find_command(text)` | Looks for a command phrase from `COMMANDS` in the text and returns its action |
| `log_note(text, action)` | Adds a line to `notes.csv` and rewrites `notes.xlsx` |

In [ ]:
COMMANDS = {                                  # phrase said: action to do
    "light on": "LIGHT ON",
    "light off": "LIGHT OFF",
    "what time": "TELL THE TIME",
    "take a note": "NOTE",
}
run = {"warned": False}


def listen_once(seconds=5):
    """Record, and turn speech into text. Returns an empty text when nothing was said."""
    x = labview.record(seconds)
    if not speech_parts(x):                   # nobody spoke: do not ask Whisper (it may invent words from noise)
        return ""
    return transcribe(x)[0]


def find_command(text):
    """The action of the first command phrase found in the text, or None."""
    spoken = " ".join(words_of(text))
    for phrase, action in COMMANDS.items():
        if phrase in spoken:
            return action
    return None


def write_excel():
    """Rewrite notes.xlsx from notes.csv, so it opens in Excel with a header."""
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font
    except ImportError:
        if not run["warned"]:
            run["warned"] = True
            print("notes.xlsx needs openpyxl. notes.csv works in Excel too. Ask your instructor.")
        return
    book = Workbook()
    sheet = book.active
    sheet.title = "Notes"
    with open("notes.csv", newline="") as f:
        for row in csv.reader(f):
            sheet.append(row)
    for cell in sheet[1]:
        cell.font = Font(bold=True)
    sheet.column_dimensions["A"].width = 20
    sheet.column_dimensions["B"].width = 60
    sheet.column_dimensions["C"].width = 16
    book.save("notes.xlsx")


def log_note(text, action):
    first = not os.path.exists("notes.csv")
    with open("notes.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "text", "action"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), text, action or ""])
    write_excel()


# the command finder can be checked without speaking
for test in ("Please turn the light on", "Turn on the light", "Turn the LIGHT OFF now.", "What time is it?", "hello there"):
    print(f"{test!r:34} ->", find_command(test))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Why is `find_command('Turn on the light')` not `LIGHT ON`, but `find_command('Please turn the light on')` is? (look at the printed result):
- How could you make the command finder understand 'turn the light on'?:


### Step 2: Run the Voice Notes
The cell listens 5 times, 5 seconds each. Say a command or any sentence each time, or stay silent to see that silence gives no text. Try: *light on*, *what time is it*, *take a note, buy batteries tomorrow*, and silence.

In [ ]:
for turn in range(5):
    print()
    print(f"[{turn + 1} of 5] >>> SPEAK NOW <<<")
    text = listen_once(5)
    if not text:
        print("(silence)")
        continue
    action = find_command(text)
    log_note(text, action)
    print("Heard:", repr(text))
    print("Action:", action or "none")

See the log. To open the Excel file, use the file browser on the left, right-click `notes.xlsx`, choose **Download** and open it on your laptop.

In [ ]:
if os.path.exists("notes.csv"):
    with open("notes.csv") as f:
        print(f.read())
else:
    print("No notes yet.")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which commands were found?:
- What did the program do for silence?:
- Did Whisper add words that you did not say? When?:


---
# Part E: Make It Your Own, Test and Present (40 min)

### Step 1: Make It Your Own
Add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run that cell again, then run Step 2 of Part D again.

- Make the command finder understand different words, for example `light on` or `turn on the light` (check that all the words are in the text, in any order).
- Add commands of your own project, for example `open the door`, `count the people`.
- Try the Malay language: `MODEL_SIZE = "base"` and `LANGUAGE = "ms"`, run Step 1 of Part C again and say a Malay sentence.
- Try `tiny.en` and `small.en`. Compare the WER and the real-time factor.
- Show the time of each speech part (from `speech_parts`) in the log.
- Record only when you press a key or a button instead of every 5 seconds.

> **Tip.** Change one thing at a time. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Which lines did you change?:
- Did it work the first time? What went wrong?:


### Step 2: Test the Accuracy in Different Situations
Say the same sentence (*please count the people at the door*) 3 times in each situation and write the WER. Use the cell to record, transcribe and calculate.

In [ ]:
TEST_SENTENCE = "please count the people at the door"
situation = "normal voice, 30 cm from the microphone"       # change this text before each test run

results = []
for turn in range(3):
    print(f">>> SPEAK NOW ({turn + 1} of 3): {TEST_SENTENCE} <<<")
    heard = transcribe(labview.record(5))[0]
    results.append(wer(TEST_SENTENCE, heard))
    print("Heard:", repr(heard), f"-> WER {100 * results[-1]:.0f} %")
print()
print(f"Situation: {situation}. Average WER: {100 * sum(results) / len(results):.0f} %")

| Test | Situation | Average WER |
|---|---|---|
| 1 | Normal voice, 30 cm from the microphone | |
| 2 | Far from the microphone (1.5 metres) | |
| 3 | With background noise (music or a talking classmate) | |
| 4 | Fast speech | |
| 5 | Whispering | |
| 6 | Another person speaks (different voice or accent) | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Average WER of test 1 to 3:
- Average WER of test 4 to 6:
- Which situation was the worst and why do you think so?:
- What would you do to get better results in a noisy place?:


### Step 3: Report and Present
Fill in the report. Then give a **3-minute demonstration**: record and show the waveform, show the speech text and the real-time factor, run the voice notes with a command, and show the log and your WER table.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Microphone used and the setting that works:
- Model, real-time factor and average WER:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- Your improvement:
- One problem you found and how you fixed it:
- One idea to make it better in future:


> **Where next.** Whisper turns **all** speech into text but it is slow for a wake word. In Lab 17 you train a small model that listens all the time for **one or two words** and reacts at once.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The microphone is set and a recording is played back | |
| 2 | The waveform is shown and speech is told from silence | |
| 3 | Whisper writes your sentence and the real-time factor is read | |
| 4 | The four checks of `wer` give the expected numbers | |
| 5 | The WER of the five sentences and the average are calculated | |
| 6 | The voice notes find a command and log it (and give no text for silence) | |
| 7 | At least one improvement works and you explain the code you changed | |
| 8 | The test table of Part E is complete | |
| 9 | The report is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. What is the sample rate and why do we use 16 000 samples per second for speech?:
- 2. How many numbers does the Jetson store for 10 seconds of sound?:
- 3. What does the real-time factor tell you? Is 1.5 good for a voice assistant?:
- 4. Calculate the WER: reference 'turn on the green light', hypothesis 'turn the green night'.:
- 5. Why does the program check for speech before it asks Whisper to transcribe?:
- 6. Name two reasons why the WER gets worse in a noisy room.:
- 7. What is the difference between `tiny.en` and `small.en`? When would you choose each?:
- 8. Why is it a benefit that Whisper runs on the Jetson and not in the cloud? Name one cost.:
- 9. Which is a good use of speech to text? (a) Writing meeting notes (b) Counting cars (c) Reading a QR code (d) Measuring distance:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `arecord is not installed` | alsa-utils is missing | Run `!sudo apt install -y alsa-utils` after `labview.sudo_login()` |
| `Cannot record: ... No such file or directory` or `Device or resource busy` | Wrong microphone name, or another program uses it | Run `labview.audio_devices()`. Set the microphone again with `labview.mic_set("plughw:CARD,DEVICE")`. Close other notebooks that record. |
| The level is always very low (-60 dB) | Wrong microphone, or the microphone is muted | Choose the right device. Check the webcam is plugged in. Speak closer. |
| The level is always very loud or the waveform is flat on top | Too close or too loud | Speak from 30 cm and not too loud |
| Nothing is heard on the speaker | The speaker is not the default one | Set it with `labview.speaker_set("plughw:CARD,DEVICE")`. The player under the cell plays on your laptop. |
| The model cannot be downloaded | No internet | Ask your teacher |
| `No module named 'faster_whisper'` | Step 1 of Part A was not run | Run it again |
| Whisper writes words from silence (like "Thank you.") | Whisper invents text from noise | Check for speech first with `speech_parts`, as in `listen_once` |
| The text is wrong | Noise, far from the microphone, wrong language model | Speak clearly from 30 cm. Use a model without `.en` for other languages. Try `small.en`. |
| It is slow | A bigger model | Use `tiny.en` or `base.en` |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(16)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- OpenAI, Whisper: https://github.com/openai/whisper
- faster-whisper: https://github.com/SYSTRAN/faster-whisper
- Word error rate: https://en.wikipedia.org/wiki/Word_error_rate
- ALSA utilities (arecord, aplay): https://www.alsa-project.org/wiki/Main_Page